In [1]:
import logging
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from spi_time_series import Pipeline, PipelineBuilder
from spi_time_series.data import Dataset
from spi_time_series.data.schemas import EventLog, PreprocessedData, RawData
from spi_time_series.evaluation import evaluate
from spi_time_series.features import extract_features
import pandas as pd

logging.basicConfig(level=logging.INFO)

## Build the pipeline

Use `PipelineBuilder` to compose a `Pipeline` from its strategy components.
Implement `my_preprocessor` and `my_splitter` below before calling `pipeline.run()`.

In [2]:
def my_preprocessor(raw: RawData) -> EventLog:
    """Clean the raw event log."""
    raise NotImplementedError


def my_splitter(log: EventLog) -> PreprocessedData:
    """Split the cleaned log into train/test sets."""
    raise NotImplementedError


pipeline = (
    PipelineBuilder()
    .with_dataset(Dataset())
    .with_preprocessor(my_preprocessor)
    .with_splitter(my_splitter)
    .with_feature_extractor(extract_features)
    .add_model("logistic_regression", LogisticRegression()) # example of adding a model
    .add_model("decision_tree", DecisionTreeClassifier()) # example of adding a model
    .add_evaluator(evaluate)
    .build()
)

INFO:spi_time_series.data.dataset:Dataset found at C:\Users\roman\Documents\Uni\Aachen\Semester_3\SoftwarePraktikum\spi-time-series\data\raw\BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


## Run the full pipeline

In [3]:
evaluation = pipeline.run()

INFO:spi_time_series.pipeline.pipeline:Pipeline started.


NotImplementedError: 

## Run stages individually

Access each strategy through the `pipeline` instance to step through stages manually.
Useful for development and debugging.

In [4]:
raw = RawData(event_log=pipeline.dataset.log)
raw.event_log.head()

,Selected,case:RequestedAmount,lifecycle:transition,Action,MonthlyCost,FirstWithdrawalAmount,EventOrigin,CreditScore,OfferID,Accepted,org:resource,case:concept:name,concept:name,OfferedAmount,case:ApplicationType,time:timestamp,EventID,NumberOfTerms,case:LoanGoal
0,None,20000.0,complete,Created,NaN,NaN,Application,NaN,None,None,User_1,Application_652823628,A_Create Application,NaN,New credit,2016-01-01 09:51:15.304000+00:00,Application_652823628,NaN,Existing loan takeover
1,None,20000.0,complete,statechange,NaN,NaN,Application,NaN,None,None,User_1,Application_652823628,A_Submitted,NaN,New credit,2016-01-01 09:51:15.352000+00:00,ApplState_1582051990,NaN,Existing loan takeover
2,None,20000.0,schedule,Created,NaN,NaN,Workflow,NaN,None,None,User_1,Application_652823628,W_Handle leads,NaN,New credit,2016-01-01 09:51:15.774000+00:00,Workitem_1298499574,NaN,Existing loan takeover
3,None,20000.0,withdraw,Deleted,NaN,NaN,Workflow,NaN,None,None,User_1,Application_652823628,W_Handle leads,NaN,New credit,2016-01-01 09:52:36.392000+00:00,Workitem_1673366067,NaN,Existing loan takeover
4,None,20000.0,schedule,Created,NaN,NaN,Workflow,NaN,None,None,User_1,Application_652823628,W_Complete application,NaN,New credit,2016-01-01 09:52:36.403000+00:00,Workitem_1493664571,NaN,Existing loan takeover


In [5]:
cleaned = pipeline.preprocessor(raw)

NotImplementedError: 

In [6]:
preprocessed = pipeline.splitter(cleaned)

NameError: name 'cleaned' is not defined

In [7]:
features = pipeline.feature_extractor(preprocessed)

NameError: name 'preprocessed' is not defined

## Preprocessing
Preprocessing does 3 steps:
1) Data Cleaning (TBD)
2) Data Splitting into train and test (TBD)
3) Prefix Generation+ Target computation

### Data Cleaning
TODO

### Data Splitting
TODO

## Prefix generation
Prefix are created and returned as generators.

In [ ]:
from spi_time_series.preprocessing.preprocess import preprocess, sliding_window_factory

# define small dataframe for quick processing
dataset = Dataset()
small_cases = dataset.log["case:concept:name"].unique()[:5]
df_small = dataset.log[dataset.log["case:concept:name"].isin(small_cases)]
raw = RawData(df_small)

INFO:spi_time_series.data.dataset:Dataset found at C:\Users\roman\Documents\Uni\Aachen\Semester_3\SoftwarePraktikum\spi-time-series\data\raw\BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
c:\Users\roman\Documents\Uni\Aachen\Semester_3\SoftwarePraktikum\spi-time-series\.venv\Lib\site-packages\pm4py\utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


In [ ]:
prefix_generator = sliding_window_factory(min_length=3, max_length=None)

def target_func(prefix: pd.DataFrame, trace: pd.DataFrame) -> int | str:
    return len(prefix)


preprocessed_data = preprocess(
    raw, target_generator=target_func, prefix_generator=prefix_generator
)

prefix_sample = next(preprocessed_data.train_log)

print(f"CASE ID: {prefix_sample.case_id}")
print(f"Target Value: {prefix_sample.target}")
print(prefix_sample.prefix.head())

CASE ID: Application_1691306052
Target Value: 31
  OfferID       case:concept:name  ... Accepted                 EventID
0    None  Application_1691306052  ...     None  Application_1691306052
1    None  Application_1691306052  ...     None     ApplState_284636842
2    None  Application_1691306052  ...     None      Workitem_831373279

[3 rows x 19 columns]
